# Task 9

Сегментация Selective Search + классификация кандидатов моделью `Comp_Vision_Task_3_model`.

In [1]:
import cv2
import numpy as np
from pathlib import Path

from keras import Model
from keras.layers import Dense
from keras.applications.vgg16 import VGG16
from keras.models import model_from_json

BASE_DIR = Path('.').resolve()

# Берем image.png, если есть; иначе используем image.jpg из папки задания
img_path = BASE_DIR / 'image.png'
if not img_path.exists():
    img_path = BASE_DIR / 'image.jpg'

model_json_path = BASE_DIR / 'Comp_Vision_Task_3_model' / 'Comp_Vision_Task_3_model.json'
model_weights_path = BASE_DIR / 'Comp_Vision_Task_3_model' / 'ieeercnn_vgg16_1.h5'

img = cv2.imread(str(img_path))
if img is None:
    raise FileNotFoundError(f'Image not found or unreadable: {img_path}')

if not hasattr(cv2, 'ximgproc'):
    raise RuntimeError('OpenCV ximgproc is unavailable. Install opencv-contrib-python.')

ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
ss.setBaseImage(img)
ss.switchToSelectiveSearchFast()
rects = ss.process()

# Сначала пробуем восстановить модель из json, как в условии.
# Для новых версий Keras используем совместимый fallback из VGG16 + Dense(2).
try:
    with open(model_json_path, 'r', encoding='utf-8') as f:
        model = model_from_json(f.read())
except Exception:
    vggmodel = VGG16(weights='imagenet', include_top=True)
    for layer in vggmodel.layers[:15]:
        layer.trainable = False
    x = vggmodel.layers[-2].output
    predictions = Dense(2, activation='softmax')(x)
    model = Model(inputs=vggmodel.input, outputs=predictions)

model.load_weights(str(model_weights_path))

threshold = 0.65
detections = []

for x, y, w, h in rects[:10000]:
    if w <= 0 or h <= 0:
        continue
    roi = img[y:y+h, x:x+w]
    if roi.size == 0:
        continue
    resized = cv2.resize(roi, (224, 224), interpolation=cv2.INTER_AREA)
    batch = np.expand_dims(resized, axis=0)
    out = model.predict(batch, verbose=0)
    score = float(out[0][0])
    if score > threshold:
        detections.append((score, int(x), int(y), int(w), int(h)))

if not detections:
    raise RuntimeError('No detections with score > 0.65')

best = max(detections, key=lambda z: z[0])
worst = min(detections, key=lambda z: z[0])

print('count =', len(detections))
print('best_score =', round(best[0], 3))
print('best_x =', best[1])
print('best_y =', best[2])
print('best_w =', best[3])
print('best_h =', best[4])
print('worst_score =', round(worst[0], 3))
print('worst_x =', worst[1])
print('worst_y =', worst[2])
print('worst_w =', worst[3])
print('worst_h =', worst[4])

print()
print('ALL DETECTIONS SORTED:')
for d in sorted(detections, key=lambda z: z[0]):
    print(round(d[0], 3), d[1], d[2], d[3], d[4])


count = 9
best_score = 0.915
best_x = 83
best_y = 71
best_w = 34
best_h = 27
worst_score = 0.662
worst_x = 78
worst_y = 71
worst_w = 36
worst_h = 46

ALL DETECTIONS SORTED:
0.662 78 71 36 46
0.76 89 71 25 27
0.805 226 155 30 33
0.808 78 71 36 26
0.813 90 71 24 26
0.844 223 153 33 35
0.849 82 71 32 27
0.857 222 154 34 34
0.915 83 71 34 27
